# 트랜스포머 파라미터 계산과 메모리 예산 - 파라미터 수 정밀 계산

- Tutorial ID: `ull-5`
- Tutorial: 트랜스포머 파라미터 계산과 메모리 예산
- Section ID: `ull-5-1`
- Section: 파라미터 수 정밀 계산

## 이 노트북에서 배우는 것
- 트랜스포머 모델이 정확히 몇 개의 파라미터(학습 가능한 숫자)로 이루어져 있는지, 구성 요소별로 직접 계산해봅니다.
- GPT-2, LLaMA 같은 실제 공개 모델의 하이퍼파라미터를 넣어보고, 우리가 계산한 값이 공식적으로 알려진 파라미터 수와 얼마나 비슷한지 검증해봅니다.
- 학습(훈련) 중에는 가중치 말고도 그래디언트, 옵티마이저 상태, 활성화값까지 메모리를 차지한다는 것을 직접 숫자로 확인합니다.
- 추론(inference) 시에는 KV 캐시라는 것이 추가로 메모리를 차지하며, 문맥 길이가 길어질수록 이게 왜 문제가 되는지 살펴봅니다.

> 이 노트북은 "정답 코드를 실행해서 숫자만 확인"하는 게 목적이 아니라, **공식이 왜 이렇게 생겼는지**를 하나씩 풀어서 이해하는 게 목적입니다. 처음 보는 용어가 나오면 바로 위/아래의 주석이나 설명을 같이 읽어주세요.


In [ ]:
# ============================================================
# 코드 읽는 법 — 파라미터 수 정밀 계산
#
# 이 코드는 "정답을 한 번 실행"하는 용도가 아니라,
# "모델 하이퍼파라미터(층 수, 차원 등) 몇 개만 알면 전체 파라미터 수와
# 메모리 사용량을 계산해낼 수 있다"는 것을 직접 손으로 짚어보기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 임베딩/어텐션/MLP/레이어노름 각각이 몇 개의 파라미터를 차지하는지,
#      그리고 그 합이 왜 그런 식으로 생겼는지 이해한다.
#   2) 우리가 만든 공식으로 GPT-2, LLaMA 같은 실제 모델의 파라미터 수를
#      계산해보고, 공식적으로 알려진 수치와 비교해 "검증"하는 습관을 들인다.
#   3) 훈련 시 메모리(가중치+그래디언트+옵티마이저+활성화값)와
#      추론 시 메모리(가중치+KV 캐시)가 왜, 그리고 얼마나 다른지 이해한다.
#
# 읽는 순서:
#   1) 각 절(섹션)의 markdown 설명을 먼저 읽고, "왜 이 숫자를 곱하는가"를
#      예상해본 다음 코드를 봅니다.
#   2) 작은 숫자로 만든 장난감(toy) 예제를 먼저 손으로 계산해보고,
#      그 다음 일반화된 함수가 같은 결과를 내는지 확인합니다.
#   3) 함수 안의 각 줄이 어떤 가중치 행렬(W_Q, W_K, W_V, W_O, MLP 등)에
#      대응되는지 주석을 따라가며 확인합니다.
#   4) 실제 모델(GPT-2, LLaMA) 하이퍼파라미터를 넣어 계산한 값을,
#      각 모델이 실제로 공개한 파라미터 수와 비교해봅니다.
#   5) d_model, n_layers, batch_size, seq_len, precision 등을 바꿔보면서
#      숫자가 어떻게, 그리고 왜 변하는지 실험해봅니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 "어떤 가중치가 몇 개의 숫자로 이루어져
#     있고, 그게 왜 그런 모양인지"를 보세요.
#   - 이 노트북은 순수 NumPy로 동작하며 별도 GPU/torch 환경이 필요 없습니다.

In [ ]:
import numpy as np

# 참고: 이 노트북은 실제로 텐서 연산을 수행하지는 않습니다.
# ("파라미터 개수"를 세는 것이라 곱셈/덧셈만 있으면 충분합니다)
# 다른 실습 노트북들과 형식을 통일하기 위해 numpy를 import해 둡니다.

print("=" * 62)
print("트랜스포머 파라미터 계산과 메모리 예산")
print("=" * 62)

## 1. 파라미터 수 정밀 계산

### "파라미터"란 무엇일까요?
신경망에서 **파라미터(parameter)** 란, 학습 과정에서 값이 조금씩 바뀌는 숫자 하나하나를 말합니다. 예를 들어 `y = w*x + b` 라는 아주 작은 식에서 `w`와 `b`가 파라미터입니다(2개). 트랜스포머는 이런 숫자(주로 가중치 행렬의 원소들)를 수억~수천억 개 가지고 있고, "이 모델은 70억 개 파라미터(7B)짜리 모델이다"라고 말할 때 바로 이 숫자를 가리킵니다.

파라미터 수가 중요한 이유:
- **메모리**: 파라미터 1개당 보통 2~4바이트가 필요합니다. 70억 개면 가중치만 14~28GB!
- **비용**: 더 많은 파라미터 = 더 많은 연산 = 더 비싼 훈련/추론
- **성능**: (다른 조건이 같다면) 보통 파라미터가 많을수록 더 많은 것을 학습할 수 있습니다

### 디코더 기반 트랜스포머의 구조 (GPT, LLaMA 계열)

```
[토큰 임베딩] + [위치 정보]
      │
      ▼
┌─────────────────────┐
│   레이어 1            │
│  - 어텐션 (Q,K,V,O)   │   ← 이 블록이
│  - LayerNorm          │      n_layers번
│  - MLP (FFN)          │      반복됩니다
│  - LayerNorm          │
└─────────────────────┘
      │
      ▼
     ...  (레이어 2, 3, ..., n_layers)
      │
      ▼
[최종 LayerNorm]
      │
      ▼
[언임베딩] → 단어별 점수(logit)
```

우리가 셀 파라미터는 정확히 이 그림에 나온 종류들입니다:
1. **토큰 임베딩**: 각 단어(토큰)를 벡터로 바꾸는 표
2. **위치 임베딩**: "이 토큰이 몇 번째 위치에 있는지"를 알려주는 정보 (학습되는 경우만)
3. **어텐션 가중치**: Q, K, V, O 네 개의 정사각 행렬, 레이어마다 있음
4. **MLP(FFN) 가중치**: 레이어마다 있는 작은 2~3층 신경망
5. **LayerNorm 파라미터**: 값을 일정한 범위로 정규화할 때 쓰는 스케일/이동 값
6. **언임베딩**: 토큰 임베딩과 같은 표를 재사용(tie)할 수도, 별도로 둘 수도 있음

지금부터 이 6가지를 하나씩 손으로 계산해보고, 마지막에 하나의 함수로 합쳐보겠습니다.

### 1-1. 토큰 임베딩 & 위치 임베딩 — 작은 숫자로 먼저 손으로 계산해보기

아주 작은 가짜 모델을 하나 상상해봅시다.
- 단어장 크기(vocab_size) = 100개의 단어만 아는 모델
- 모델 차원(d_model) = 8 (각 단어를 8개의 숫자로 표현)
- 최대 문장 길이(max_seq) = 16

**토큰 임베딩**은 "단어 100개 × 각 단어를 8개의 숫자로" 표현하는 표이므로, 100×8 = 800개의 숫자(파라미터)가 필요합니다.

**위치 임베딩**(학습 가능한 절대 위치 임베딩을 쓰는 경우, 예: GPT-2)도 똑같은 방식입니다 — "가능한 위치 16개 × 각 위치를 8개의 숫자로" → 16×8 = 128개.

아래 코드로 직접 확인해봅니다.

In [ ]:
print("1. 파라미터 수 정밀 계산")
print("-" * 50)

# --- 토이(장난감) 예제: 아주 작은 가짜 모델 ---
toy_vocab = 100       # 단어장에 100개의 단어만 있다고 가정
toy_d_model = 8        # 각 단어를 8개의 숫자(벡터)로 표현
toy_max_seq = 16       # 최대 16개의 토큰까지 위치를 구분

toy_token_embed = toy_vocab * toy_d_model
toy_pos_embed = toy_max_seq * toy_d_model

print(f"  토이 모델: vocab={toy_vocab}, d_model={toy_d_model}, max_seq={toy_max_seq}")
print(f"  토큰 임베딩 파라미터 수 = vocab × d_model = {toy_vocab} × {toy_d_model} = {toy_token_embed}")
print(f"  위치 임베딩 파라미터 수 = max_seq × d_model = {toy_max_seq} × {toy_d_model} = {toy_pos_embed}")

### 1-2. 어텐션 가중치 — 왜 `4 × d_model × d_model`일까요?

멀티헤드 어텐션은 입력을 Query(Q), Key(K), Value(V) 세 가지로 투영(projection)한 다음, 어텐션을 계산하고, 마지막에 Output(O) 행렬로 다시 투영합니다. 네 개의 가중치 행렬 W_Q, W_K, W_V, W_O는 **각각** `d_model × d_model` 크기입니다 (입력도 d_model차원, 출력도 d_model차원이기 때문입니다).

> 헷갈리기 쉬운 부분: "헤드가 여러 개면 파라미터도 늘어나지 않을까?" → **아닙니다.** 헤드 수(n_heads)는 d_model을 "몇 조각으로 나눠서 병렬로 어텐션을 계산하느냐"만 결정할 뿐, 행렬 자체의 전체 크기는 바뀌지 않습니다. 예를 들어 d_model=8인데 헤드가 2개면 각 헤드는 4차원(d_head = d_model / n_heads = 4)을 보지만, 4개 헤드의 결과를 다시 이어붙이면(concat) 결국 8차원이 되고, W_Q 자체는 여전히 8×8짜리 행렬 하나입니다. 즉 **헤드 수는 "어떻게 나눠서 보느냐"의 문제이고, 파라미터 수는 d_model에만 의존합니다.**

그래서 어텐션 한 레이어의 파라미터 수는:
```
W_Q + W_K + W_V + W_O = 4 × (d_model × d_model)
```
여기에 만약 각 투영에 bias(절편)를 추가한다면(GPT-2 계열), bias는 출력 차원만큼(d_model개)씩 4개 더 추가됩니다: `+ 4 × d_model`.

토이 모델(d_model=8, n_heads=2)로 직접 확인해봅니다.

In [ ]:
toy_n_heads = 2
toy_d_head = toy_d_model // toy_n_heads  # 헤드 하나가 보는 차원

toy_attn_no_bias = 4 * toy_d_model * toy_d_model
toy_attn_bias = 4 * toy_d_model  # W_Q, W_K, W_V, W_O 각각의 bias

print(f"  헤드 수 = {toy_n_heads}, 헤드당 차원(d_head) = d_model / n_heads = {toy_d_head}")
print(f"  → 참고: d_head는 '나눠서 보는 방식'만 결정하고, 아래 파라미터 수 계산에는 등장하지 않습니다.")
print(f"  W_Q/W_K/W_V/W_O (bias 없음) = 4 × {toy_d_model} × {toy_d_model} = {toy_attn_no_bias}")
print(f"  bias 4개 추가 시 = +{toy_attn_bias} → 총 {toy_attn_no_bias + toy_attn_bias}")

# 헤드 수를 4개로 바꿔도 파라미터 수는 그대로인지 확인
toy_n_heads_alt = 4
toy_attn_no_bias_alt = 4 * toy_d_model * toy_d_model
print(f"\n  헤드 수를 {toy_n_heads_alt}개로 바꿔도: 4 × {toy_d_model} × {toy_d_model} = {toy_attn_no_bias_alt} (동일!)")

### 1-3. MLP(FFN) 가중치 — 표준 방식 vs SwiGLU 방식

어텐션 다음에는 각 레이어마다 작은 2층(또는 3층) 신경망인 MLP(Feed-Forward Network)가 있습니다. 보통 d_model보다 훨씬 넓은 중간 차원(d_ff, 보통 d_model의 4배 정도)으로 한 번 넓혔다가 다시 d_model로 좁힙니다.

**표준 방식 (GPT-2 등)**: "올리고(up) → 내리고(down)" 2개의 행렬
```
up:   d_model → d_ff     →  d_model × d_ff 개의 숫자
down: d_ff → d_model     →  d_ff × d_model 개의 숫자
합계 = 2 × d_model × d_ff
```

**SwiGLU 방식 (LLaMA 등)**: 행렬이 1개 더 있습니다 — gate, up, down 3개
```
gate: d_model → d_ff
up:   d_model → d_ff
down: d_ff → d_model
합계 = 3 × d_model × d_ff
```
SwiGLU는 게이트(gate) 행렬을 하나 더 써서 표현력을 높이는 대신 행렬이 1개 늘어납니다. 그래서 LLaMA 같은 모델들은 이를 보정하려고 **d_ff를 표준 방식(4×d_model)보다 작게**(대략 2/3 × 4 × d_model) 잡아서, 전체 연산량/파라미터 수를 표준 방식과 비슷하게 맞춥니다. (실제로 LLaMA-7B는 d_model=4096이라 표준 방식이면 d_ff=4×4096=16384가 되지만, 실제 d_ff는 11008로 더 작습니다.)

bias를 쓰는 경우(GPT-2): up의 출력(d_ff개)과 down의 출력(d_model개) 각각에 bias가 붙어 `+ (d_ff + d_model)`이 추가됩니다.

In [ ]:
toy_d_ff_standard = 4 * toy_d_model  # 표준 방식: 보통 4배
toy_mlp_standard = 2 * toy_d_model * toy_d_ff_standard

toy_d_ff_swiglu = int(2/3 * 4 * toy_d_model)  # SwiGLU는 보정해서 더 작게
toy_mlp_swiglu = 3 * toy_d_model * toy_d_ff_swiglu

print(f"  [표준 방식] d_ff = 4 × d_model = {toy_d_ff_standard}")
print(f"             MLP 파라미터 = 2 × {toy_d_model} × {toy_d_ff_standard} = {toy_mlp_standard}")
print(f"  [SwiGLU]   d_ff = 약 2/3 × 4 × d_model = {toy_d_ff_swiglu} (더 작게 보정)")
print(f"             MLP 파라미터 = 3 × {toy_d_model} × {toy_d_ff_swiglu} = {toy_mlp_swiglu}")
print(f"  → d_ff를 줄여서 보정했기 때문에 두 방식의 파라미터 수가 비슷해집니다 ({toy_mlp_standard} vs {toy_mlp_swiglu}).")

### 1-4. LayerNorm 파라미터, 그리고 "임베딩 공유(tie)"

**LayerNorm**은 각 레이어를 통과한 값을 평균 0, 분산 1 근처로 정규화한 뒤, 학습 가능한 두 벡터로 다시 스케일/이동시킵니다:
```
출력 = γ(스케일) × 정규화된 값 + β(이동)
```
γ, β는 각각 d_model개의 숫자입니다. 트랜스포머 한 레이어에는 보통 LayerNorm이 2번 등장합니다(어텐션 앞/뒤, MLP 앞/뒤 등 구현에 따라 위치는 다르지만 "2번"은 거의 공통입니다). 그래서 레이어당 `2(γ,β) × 2(개수) × d_model`개의 파라미터가 필요합니다. 그리고 맨 마지막, 언임베딩 직전에 최종 LayerNorm이 1번 더 있습니다(+ `2 × d_model`).

> 참고: LLaMA 같은 모델은 LayerNorm 대신 **RMSNorm**을 쓰는데, RMSNorm은 β(이동) 없이 γ(스케일)만 있어서 LayerNorm의 절반 파라미터만 씁니다. 이 노트북의 기본 공식은 "표준 LayerNorm" 기준이라, LLaMA류 모델에는 아주 약간(전체에서 차지하는 비중이 0.01%도 안 되는 수준) 더 많게 계산될 수 있습니다 — 이런 작은 차이까지 알아두면 "내가 계산한 값이 왜 공식 발표 수치와 살짝 다른가"를 스스로 설명할 수 있습니다.

**임베딩 공유(weight tying)**: 입력의 "토큰 → 벡터" 표(토큰 임베딩, vocab×d_model)와 출력의 "벡터 → 단어별 점수" 표(언임베딩, d_model×vocab)는 크기가 똑같습니다. 그래서 많은 모델(GPT-2 등)은 **이 둘을 같은 행렬로 공유**해서 파라미터를 절약합니다(`tie_embeddings=True` → 언임베딩 추가 파라미터 0개). LLaMA처럼 공유하지 않는 모델은 언임베딩 행렬을 통째로 하나 더 가집니다.

In [ ]:
toy_ln_per_layer = 2 * 2 * toy_d_model   # 레이어당 LayerNorm 2번 × (γ,β) 2개
toy_ln_final = 2 * toy_d_model           # 마지막 최종 LayerNorm 1번

print(f"  레이어 1개당 LayerNorm 파라미터 = 2(γ,β) × 2(개수) × d_model = 2×2×{toy_d_model} = {toy_ln_per_layer}")
print(f"  최종 LayerNorm 파라미터 = 2 × d_model = {toy_ln_final}")

toy_unembed_tied = 0
toy_unembed_untied = toy_d_model * toy_vocab
print(f"\n  임베딩 공유(tie) 시 언임베딩 추가 파라미터 = {toy_unembed_tied}")
print(f"  공유하지 않을 시 언임베딩 파라미터 = d_model × vocab = {toy_d_model} × {toy_vocab} = {toy_unembed_untied}")

### 1-5. 이제 전부 합쳐서 함수로 만들기

위에서 손으로 계산해본 6가지 조각(토큰 임베딩, 위치 임베딩, 어텐션, MLP, LayerNorm, 언임베딩)을 그대로 하나의 함수 `count_parameters`로 옮깁니다. 함수 안의 각 줄에 주석으로 "이게 위에서 계산한 어떤 조각인지" 표시해두었으니, 위 설명과 비교하면서 읽어보세요.

In [ ]:
def count_parameters(vocab_size, d_model, n_layers, n_heads, d_ff,
                      max_seq=2048, tie_embeddings=True, use_swiglu=False,
                      use_bias=True):
    """
    트랜스포머(디코더 전용, GPT/LLaMA 계열) 파라미터 수를 구성 요소별로 정밀 계산합니다.

    매개변수
    ----------
    vocab_size : 단어장(토큰) 크기
    d_model    : 모델의 은닉 차원 (임베딩 벡터 길이)
    n_layers   : 트랜스포머 블록(레이어) 개수
    n_heads    : 어텐션 헤드 개수 (※ 위에서 봤듯, 파라미터 수 자체에는 영향 없음.
                 d_head = d_model // n_heads 계산을 보여주기 위해서만 받습니다)
    d_ff       : MLP(FFN)의 중간(확장) 차원
    max_seq    : 학습 가능한 위치 임베딩을 쓸 때, 다룰 수 있는 최대 토큰 위치 수
    tie_embeddings : True면 토큰 임베딩과 언임베딩을 같은 행렬로 공유
    use_swiglu : True면 LLaMA 스타일 SwiGLU MLP(행렬 3개), False면 표준 MLP(행렬 2개)
    use_bias   : True면 어텐션/MLP의 각 투영에 bias(절편)도 추가

    반환값
    ----------
    각 구성 요소별 파라미터 수가 담긴 dict.
    (총 파라미터 수가 필요하면 sum(결과.values())로 더하면 됩니다)
    """
    params = {}

    # ── 1) 토큰 임베딩 ───────────────────────────────────────
    # "단어 vocab_size개 × 각 단어를 d_model개의 숫자로 표현"하는 표
    params['token_embed'] = vocab_size * d_model

    # ── 2) 위치 임베딩 ───────────────────────────────────────
    # 학습 가능한 절대 위치 임베딩(GPT-2 스타일)을 가정합니다.
    # (RoPE처럼 학습되지 않는 위치 인코딩을 쓰는 모델은 실제로는 0개입니다 —
    #  이 함수는 단순화를 위해 항상 이 항을 더하므로, LLaMA류 모델에서는
    #  아래 표에서 실제보다 "위치 임베딩"이 조금 더 크게 잡힙니다)
    params['pos_embed'] = max_seq * d_model

    # ── 3) 어텐션 (레이어 1개 기준) ─────────────────────────
    d_head = d_model // n_heads
    # ↑ 위 1-2절에서 본 것처럼, d_head는 "헤드가 d_model을 몇 조각으로
    #   나눠 보는지"를 보여주기 위해 계산만 해둘 뿐, 아래 파라미터 수
    #   계산식에는 등장하지 않습니다 (W_Q/W_K/W_V/W_O 전체 크기는
    #   헤드 수와 무관하게 d_model × d_model 이기 때문).
    attn_per_layer = 4 * d_model * d_model  # W_Q, W_K, W_V, W_O
    if use_bias:
        attn_per_layer += 4 * d_model  # 4개 투영 각각의 bias
    params['attention'] = attn_per_layer * n_layers  # n_layers개 레이어 전체

    # ── 4) MLP (레이어 1개 기준) ────────────────────────────
    if use_swiglu:
        # SwiGLU: gate, up, down — 행렬 3개 (LLaMA 등)
        mlp_per_layer = 3 * d_model * d_ff
    else:
        # 표준 MLP: up, down — 행렬 2개 (GPT-2 등)
        mlp_per_layer = 2 * d_model * d_ff
    if use_bias:
        mlp_per_layer += d_ff + d_model  # up 출력(d_ff) + down 출력(d_model) bias
    params['mlp'] = mlp_per_layer * n_layers

    # ── 5) LayerNorm ─────────────────────────────────────────
    # 레이어마다 LayerNorm 2번, 각각 (γ, β) 2개의 벡터 → 2 × 2 × d_model
    ln_per_layer = 2 * 2 * d_model
    params['layer_norm'] = ln_per_layer * n_layers + 2 * d_model  # + 마지막 최종 LayerNorm 1번

    # ── 6) 언임베딩 ──────────────────────────────────────────
    if tie_embeddings:
        params['unembed'] = 0  # 토큰 임베딩 행렬을 그대로 재사용 (파라미터 절약)
    else:
        params['unembed'] = d_model * vocab_size  # 별도의 d_model × vocab_size 행렬

    return params

### 1-6. 실제 공개 모델에 대입해서 검증해보기

이제 GPT-2 계열(표준 MLP + 절대 위치 임베딩 + bias + 임베딩 공유)과 LLaMA 계열(SwiGLU + RoPE + bias 없음 + 임베딩 비공유)의 실제 하이퍼파라미터를 넣어봅니다. 표에서 "공식 발표" 값과 "우리 계산" 값이 얼마나 비슷한지 비교해보는 것이 이 절의 핵심입니다.

In [ ]:
# 실제 공개된 모델들의 하이퍼파라미터
# (vocab_size, d_model, n_layers, n_heads, d_ff, max_seq, tie_embeddings, use_swiglu, use_bias)
models = [
    ("GPT-2 Small",   50257, 768,  12, 12, 3072,  1024, True,  False, True),
    ("GPT-2 Medium",  50257, 1024, 24, 16, 4096,  1024, True,  False, True),
    ("GPT-2 Large",   50257, 1280, 36, 20, 5120,  1024, True,  False, True),
    ("GPT-2 XL",      50257, 1600, 48, 25, 6400,  1024, True,  False, True),
    ("LLaMA-7B",      32000, 4096, 32, 32, 11008, 2048, False, True,  False),
    ("LLaMA-13B",     32000, 5120, 40, 40, 13824, 2048, False, True,  False),
    ("LLaMA-70B",     32000, 8192, 80, 64, 28672, 4096, False, True,  False),
]

# 비교용: 각 모델이 실제로 공개 발표한(논문/공식 자료 기준) 대략적인 파라미터 수
# (모델명에 들어간 "7B", "70B" 같은 숫자는 보통 반올림된 마케팅용 숫자라
#  실제 정밀한 값과 살짝 다를 수 있습니다)
official_params_B = {
    "GPT-2 Small":  0.124,
    "GPT-2 Medium": 0.355,
    "GPT-2 Large":  0.774,
    "GPT-2 XL":     1.5,   # 정밀하게는 약 1.557B (1.5B는 반올림된 표기)
    "LLaMA-7B":     6.7,
    "LLaMA-13B":    13.0,
    "LLaMA-70B":    68.9,  # LLaMA-2 70B. 아래 1-7절에서 왜 우리 계산과 차이가 나는지 설명합니다
}

print(f"{'Model':>14}  {'우리 계산':>10}  {'공식 발표':>10}  {'Embed':>10}  {'Attn':>10}  {'MLP':>10}  {'LN':>8}")
print("  " + "-" * 88)

for name, V, d, L, H, ff, seq, tie, swi, bias in models:
    p = count_parameters(V, d, L, H, ff, seq, tie, swi, bias)
    total = sum(p.values())

    print(f"  {name:>12}  {total/1e9:>8.2f}B  {official_params_B[name]:>8.2f}B  "
          f"{p['token_embed']/1e6:>7.1f}M  {p['attention']/1e6:>7.1f}M  "
          f"{p['mlp']/1e6:>7.1f}M  {p['layer_norm']/1e3:>5.0f}K")

### 1-7. 결과 해석 — 왜 GPT-2는 거의 정확하고, LLaMA-70B는 차이가 날까요?

GPT-2 계열(Small~XL)은 "우리 계산"과 "공식 발표" 값이 거의 똑같습니다. GPT-2는 우리가 가정한 구조(표준 LayerNorm, 학습되는 절대 위치 임베딩, 표준 MLP)와 실제 구조가 정확히 일치하기 때문입니다.

LLaMA-7B, 13B도 거의 정확합니다 — RoPE(위치 임베딩 0개)와 RMSNorm(LayerNorm보다 살짝 적은 파라미터) 때문에 생기는 차이가 전체 대비 0.1% 수준으로 작아서 거의 묻힙니다.

그런데 **LLaMA-70B(정확히는 LLaMA-2 70B)는 우리 계산이 실제보다 꽤 크게 나옵니다.** 이유는 이 모델이 **GQA(Grouped-Query Attention, 그룹 쿼리 어텐션)**라는 기법을 쓰기 때문입니다:
- 표준 멀티헤드 어텐션: Q, K, V 모두 64개의 헤드를 가짐 → K, V도 Q만큼 큼
- GQA: Q는 64개 헤드를 유지하지만, **K와 V는 8개 헤드만** 가짐 (여러 Q헤드가 K,V 헤드를 그룹으로 공유)
- 그 결과 K, V를 만드는 가중치 행렬이 Q, O보다 훨씬 작아져서, 어텐션 전체 파라미터가 우리가 가정한 `4 × d_model × d_model`보다 적어집니다

GQA, 그리고 K·V를 헤드 1개로 극단적으로 줄이는 MQA(Multi-Query Attention)는 최근 대형 모델(LLaMA-2 70B, LLaMA-3, Mistral 등)에서 널리 쓰이는데, 핵심 이유는 **추론 시 KV 캐시 메모리를 줄이기 위해서**입니다 — 이건 3절에서 다시 다룹니다.

> 정리: "정밀 계산"이라는 제목이 붙어 있어도, 모델 구조가 우리가 가정한 것과 다르면(RoPE, RMSNorm, GQA 등) 100% 정확히 맞아떨어지지는 않습니다. 그래도 **자릿수(몇 B인지)와 대략적인 비율은 항상 맞기 때문에, 모델을 설계하는 단계에서 "이 정도 하이퍼파라미터면 대략 몇 B짜리 모델이 되겠다"를 빠르게 가늠하는 데는 충분히 유용합니다.**

## 2. 훈련 시 메모리 예산

모델을 훈련시킬 때 GPU 메모리에는 **가중치만** 올라가는 게 아닙니다. 보통 다음 4가지가 함께 올라갑니다:

| 구성 요소 | 설명 | 크기 (대략) |
|---|---|---|
| 가중치 (weights) | 모델 파라미터 자체 | 파라미터 수 × (정밀도별 바이트) |
| 그래디언트 (gradients) | 역전파로 계산된, 각 파라미터를 "얼마나/어느 방향으로" 바꿀지 | 가중치와 같은 크기 |
| 옵티마이저 상태 (optimizer state) | Adam이 각 파라미터마다 따로 기억하는 "이동평균" 2개(m, v) | 보통 항상 FP32로 유지 |
| 활성화값 (activations) | 순전파 중 각 레이어에서 만들어지는 중간 결과값 (역전파를 위해 저장해둠) | 배치 크기·문장 길이에 비례 |

추가로 **FP16/BF16 혼합 정밀도(mixed precision)** 훈련을 쓰면, 속도를 위해 가중치/그래디언트는 16비트로 다루지만, 옵티마이저가 정확하게 업데이트하도록 **FP32 "마스터 가중치"**를 별도로 한 벌 더 유지하는 경우가 많습니다.

> **Adam 옵티마이저가 왜 파라미터마다 숫자를 2개 더 저장하나요?**
> Adam은 단순히 "기울기 방향으로 한 걸음 이동"하는 게 아니라, 각 파라미터별로 "최근 기울기들의 평균(m, 1차 모멘트)"과 "최근 기울기 크기의 평균(v, 2차 모멘트)"을 계속 추적하면서 업데이트 크기를 자동으로 조절합니다. 그래서 파라미터 1개당 가중치 값 외에 m, v 두 개를 추가로 저장해야 하고, 이 두 값은 안정적인 학습을 위해 정밀도를 낮추지 않고 보통 **항상 FP32**로 유지합니다.

In [ ]:
def training_memory(n_params, batch_size, seq_len, d_model, n_layers,
                     precision='fp32'):
    """
    훈련 시 GPU에 올라가는 총 메모리 사용량을 추정합니다.
    (실제 프레임워크/구현에 따라 세부 수치는 다를 수 있는 "어림 계산"입니다)
    """
    # 정밀도(precision)에 따라 숫자 하나를 표현하는 데 필요한 바이트 수가 다릅니다.
    # fp32: 4바이트(가장 정밀하지만 가장 큼), fp16/bf16: 2바이트(절반 크기)
    if precision == 'fp32':
        bytes_per_param = 4
    elif precision == 'fp16':
        bytes_per_param = 2
    elif precision == 'bf16':
        bytes_per_param = 2
    else:
        bytes_per_param = 4

    mem = {}

    # 1) 모델 가중치: 파라미터 1개 × bytes_per_param
    mem['weights'] = n_params * bytes_per_param

    # 2) 그래디언트: 가중치와 정확히 같은 모양/같은 정밀도로 1개씩 더 필요
    mem['gradients'] = n_params * bytes_per_param

    # 3) Adam 옵티마이저 상태 (m, v) — 학습 안정성을 위해 정밀도를 낮추지 않고
    #    항상 FP32(4바이트)로 유지하는 것이 일반적입니다. m, v 두 개니까 ×2.
    mem['optimizer'] = n_params * 4 * 2

    # 4) FP32 마스터 가중치: 혼합 정밀도(fp16/bf16) 훈련에서만 필요합니다.
    #    16비트 가중치로는 아주 작은 업데이트 값이 반올림되어 사라질 수 있어서,
    #    옵티마이저가 실제로 업데이트를 적용하는 "원본"은 FP32로 따로 보관합니다.
    if precision in ('fp16', 'bf16'):
        mem['master_weights'] = n_params * 4
    else:
        mem['master_weights'] = 0  # 이미 FP32라서 마스터 사본이 따로 필요 없음

    # 5) 활성화값: 역전파(backward) 때 필요한 순전파 중간 결과들.
    #    레이어마다 "대략 배치 × 문장길이 × d_model개의 숫자가 10벌쯤 저장된다"는
    #    경험적인 어림값입니다 (Q,K,V, 어텐션 출력, MLP 중간값 등을 다 합친 대략치).
    #    ※ activation checkpointing(필요할 때 다시 계산)을 쓰면 이 값을 크게 줄일
    #      수 있지만, 여기서는 가장 단순한(체크포인팅 없는) 경우를 가정합니다.
    mem['activations'] = 10 * batch_size * seq_len * d_model * bytes_per_param * n_layers

    return mem

### 2-1. GPT-2 모델들로 실제 메모리 계산해보기

아래에서는 GPT-2 4종에 대해 배치 크기 4, 문장 길이 1024 기준으로 훈련 메모리를 계산합니다. 이번에는 FP32 한 가지만 보지 않고, **FP32 vs FP16 vs BF16**을 나란히 비교해서 "혼합 정밀도가 실제로 무엇을 절약해주는지"를 직접 확인해봅니다.

In [ ]:
print("\n\n2. 훈련 시 메모리 예산")
print("-" * 50)

for name, V, d, L, H, ff, seq, tie, swi, bias in models[:4]:  # GPT-2만
    p = count_parameters(V, d, L, H, ff, seq, tie, swi, bias)
    n_params = sum(p.values())

    print(f"\n  {name} (batch=4, seq=1024 기준, 파라미터 {n_params/1e9:.2f}B):")
    for precision in ['fp32', 'fp16', 'bf16']:
        mem = training_memory(n_params, batch_size=4, seq_len=1024, d_model=d, n_layers=L,
                               precision=precision)
        total_gb = sum(mem.values()) / 1e9
        static_gb = (mem['weights'] + mem['gradients'] + mem['optimizer'] + mem['master_weights']) / 1e9
        print(f"    [{precision:>4}] 가중치 {mem['weights']/1e9:5.2f}GB + 그래디언트 {mem['gradients']/1e9:5.2f}GB "
              f"+ 옵티마이저 {mem['optimizer']/1e9:5.2f}GB + 마스터 {mem['master_weights']/1e9:5.2f}GB "
              f"+ 활성화값 {mem['activations']/1e9:5.2f}GB = 총 {total_gb:5.2f}GB  "
              f"(파라미터 관련 고정분: {static_gb:5.2f}GB)")

### 2-2. 흥미로운 관찰 — 혼합 정밀도는 정말 "가중치" 메모리를 줄여줄까요?

위 출력에서 "파라미터 관련 고정분"(가중치+그래디언트+옵티마이저+마스터가중치의 합)을 fp32, fp16, bf16끼리 비교해보세요. **거의 똑같습니다.** 왜 그럴까요?

- FP32: 가중치(4B) + 그래디언트(4B) + 옵티마이저(8B) + 마스터(0B) = **파라미터당 16바이트**
- FP16/BF16: 가중치(2B) + 그래디언트(2B) + 옵티마이저(8B) + 마스터(4B) = **파라미터당 16바이트**

가중치·그래디언트를 16비트로 줄여서 4바이트를 절약하지만, 그 절약분을 FP32 마스터 가중치(+4바이트)가 정확히 상쇄해버립니다! 이 "파라미터당 16바이트" 어림값은 실제로 혼합 정밀도 Adam 훈련의 메모리를 추정할 때 실무에서도 자주 인용되는 수치입니다.

그렇다면 혼합 정밀도는 메모리 면에서는 의미가 없는 걸까요? 아닙니다 — 위 표를 보면 **활성화값(activations)** 은 정밀도에 정확히 비례해서 줄어듭니다(fp32의 절반). 실제 대규모 훈련에서는 활성화값이 차지하는 비중이 크기 때문에(특히 배치 크기·문장 길이가 클 때), 혼합 정밀도의 메모리 절약 효과는 주로 **이 활성화값에서** 나옵니다. (속도 면에서는 16비트 연산이 하드웨어에서 훨씬 빠르다는 별개의 큰 이점도 있습니다.)

## 3. 추론 시 메모리 (KV 캐시 포함)

훈련이 끝나고 나면 그래디언트와 옵티마이저 상태는 더 이상 필요 없습니다 — 그래서 추론(inference) 시 메모리는 훈련보다 훨씬 단순합니다:
```
추론 메모리 = 가중치 + KV 캐시 (+ 약간의 활성화값, 보통 무시할 정도로 작음)
```

### KV 캐시란 무엇인가요?

언어모델은 토큰을 한 개씩 순서대로 생성합니다(자기회귀, autoregressive). 새 토큰을 만들 때마다 어텐션은 "지금까지 나온 모든 토큰의 K(Key), V(Value)"가 필요합니다. 그런데 이전 토큰들의 K, V는 한 번 계산하면 절대 바뀌지 않습니다 — 매번 새 토큰을 만들 때마다 처음부터 다시 계산하면 엄청난 낭비입니다.

그래서 **이전 토큰들의 K, V를 메모리에 저장해두고 재사용**합니다. 이걸 KV 캐시(KV cache)라고 부릅니다. 새 토큰이 생길 때마다 그 토큰의 K, V만 새로 계산해서 캐시에 추가하면 됩니다.

문제는, 문장이 길어질수록(seq_len 증가) 캐시해야 할 K, V 개수도 계속 늘어나서, **문맥이 길어질수록 KV 캐시가 점점 더 많은 메모리를 차지**한다는 점입니다. (앞 절에서 언급한 GQA/MQA가 인기 있는 이유가 바로 이것입니다 — K, V의 헤드 수를 줄이면 캐시 크기도 그만큼 줄어듭니다.)

KV 캐시 크기 계산:
```
레이어 1개, 토큰 1개당 캐시할 숫자 개수 = 2(K와 V) × n_heads × d_head = 2 × d_model
전체 캐시 크기 = 2 × n_layers × seq_len × d_model × bytes_per_param
```

In [ ]:
print("\n\n3. 추론 시 메모리 (KV 캐시 포함)")
print("-" * 50)

for name, V, d, L, H, ff, seq, tie, swi, bias in models:
    n_params = sum(count_parameters(V, d, L, H, ff, seq, tie, swi, bias).values())
    d_head = d // H  # 헤드 하나가 담당하는 차원 (KV 캐시 크기 계산에 사용)

    # 가중치 메모리 (추론은 보통 FP16으로 서빙 — 2바이트/파라미터)
    weight_mem = n_params * 2

    # KV 캐시 메모리 (FP16 기준)
    #   2          : K와 V, 두 가지를 저장하므로
    #   × L        : 레이어마다 각자의 K, V가 있으므로
    #   × seq      : 지금까지의 토큰 수만큼
    #   × H × d_head (= d_model) : 헤드들을 합친 전체 벡터 길이
    #   × 2        : FP16 = 2바이트
    # ※ 이 줄은 "문장 1개(batch_size=1)" 기준입니다. 배치로 여러 문장을
    #   동시에 처리하면 batch_size를 한 번 더 곱해야 합니다.
    kv_cache = 2 * L * seq * H * d_head * 2

    total = (weight_mem + kv_cache) / 1e9
    print(f"  {name:>12}: 가중치 {weight_mem/1e9:.1f}GB + KV캐시(seq={seq}) {kv_cache/1e9:.2f}GB = {total:.1f}GB")

### 3-1. 문맥 길이가 길어지면 KV 캐시는 얼마나 커질까요?

위 계산은 모델별로 "각 모델이 원래 지원하는 최대 길이(max_seq)" 하나만 봤습니다. 이번에는 **LLaMA-7B 하나**를 고정해두고, 문맥 길이(seq_len)를 2K → 8K → 32K → 128K로 늘려가면서 KV 캐시가 얼마나 커지는지 직접 확인해봅니다. (요즘 "긴 문맥(long-context)" 모델을 서빙하기 어려운 이유를 체감할 수 있습니다.)

In [ ]:
llama7b = ("LLaMA-7B", 32000, 4096, 32, 32, 11008, 2048, False, True, False)
name, V, d, L, H, ff, seq, tie, swi, bias = llama7b
n_params = sum(count_parameters(V, d, L, H, ff, seq, tie, swi, bias).values())
d_head = d // H
weight_mem_gb = (n_params * 2) / 1e9  # FP16 가중치, 문맥 길이와 무관하게 고정

print(f"  {name} 가중치 메모리(FP16, 고정): {weight_mem_gb:.1f}GB\n")
print(f"  {'문맥 길이':>10}  {'KV캐시 (batch=1)':>16}  {'KV캐시 (batch=8)':>16}")
print("  " + "-" * 50)

for context_len in [2048, 8192, 32768, 131072]:
    kv_cache_b1 = (2 * L * context_len * H * d_head * 2) / 1e9
    kv_cache_b8 = kv_cache_b1 * 8   # 배치 크기 8 → 동시에 8개 문장을 캐시
    print(f"  {context_len:>10,}  {kv_cache_b1:>14.1f}GB  {kv_cache_b8:>14.1f}GB")

print(f"\n  → 문맥 길이가 64배(2K→128K) 늘어나면 KV 캐시도 정확히 64배 커집니다.")
print(f"  → 배치 크기를 늘리는 것도 KV 캐시를 그만큼 곱해서 늘립니다.")
print(f"  → 가중치({weight_mem_gb:.1f}GB)는 고정인데, 128K 문맥 + batch=8에서는")
print(f"    KV 캐시만으로도 가중치보다 훨씬 큰 메모리가 필요해질 수 있습니다!")

## 정리

- **파라미터 수**는 "어떤 행렬이 몇 개 있고, 각 행렬이 어떤 크기인지"만 알면 곱셈/덧셈으로 정확히(또는 거의 정확히) 계산할 수 있습니다. 어텐션은 `4 × d_model²`(헤드 수와 무관!), MLP는 표준이면 `2 × d_model × d_ff`, SwiGLU면 `3 × d_model × d_ff`가 핵심 공식입니다.
- 실제 모델과 비교했을 때, RoPE·RMSNorm·GQA 같은 "최신 기법"들은 우리의 단순한 공식과 살짝 다른 결과를 만듭니다 — 이런 차이를 알아두면 각 기법이 왜 쓰이는지(주로 메모리/효율 때문)도 함께 이해할 수 있습니다.
- **훈련 메모리**는 가중치 하나가 아니라 가중치+그래디언트+옵티마이저+(마스터가중치)+활성화값을 모두 더해야 하며, 흔히 "파라미터당 16바이트(혼합 정밀도 Adam 기준)"로 어림잡습니다.
- **추론 메모리**는 가중치 + KV 캐시이며, KV 캐시는 문맥 길이와 배치 크기에 정확히 비례해서 커집니다 — 이것이 긴 문맥을 다루는 LLM 서빙이 어려운 핵심 이유 중 하나입니다.

### 더 해볼 수 있는 실험
- `count_parameters`에 직접 새 모델(예: Mistral-7B, Qwen 등)의 하이퍼파라미터를 넣어 우리 계산이 공식 발표치와 얼마나 가까운지 확인해보세요.
- `training_memory`에서 batch_size나 seq_len을 키워보면서 "활성화값"이 전체 메모리에서 차지하는 비중이 어떻게 변하는지 관찰해보세요.
- GQA를 직접 반영하도록 `count_parameters`나 KV 캐시 계산식을 수정해보세요 (힌트: K, V 투영 행렬 크기를 `d_model × (n_kv_heads × d_head)`로 바꾸면 됩니다).